[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter1/sample-rag.ipynb)

# Chapter 1: Simple RAG Example with LangChain

This notebook demonstrates the basic flow of Retrieval-Augmented Generation (RAG) using LangChain:
1. **Load**: Download and parse a PDF document
2. **Split**: Chunk the document into smaller pieces
3. **Embed & Store**: Create embeddings and store in a vector database
4. **Query**: Retrieve relevant chunks and generate responses

**What you'll learn:**
- Build a basic RAG pipeline from scratch using LangChain
- Load, split, embed, and store PDF documents in a vector database (LanceDB)
- Create a retrieval-augmented generation chain using LCEL
- Query the system and get context-grounded answers

**Prerequisites:** `pip install -r requirements.txt` and set `GOOGLE_API_KEY` in your environment.

In [16]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    RunnableSequence,
)
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStoreRetriever
from pydantic import SecretStr

# Read the key once here and pass it to both Google clients. Without this, each
# client reads GOOGLE_API_KEY on its own and silently falls back to GEMINI_API_KEY.
if "GOOGLE_API_KEY" not in os.environ:
    raise RuntimeError("Set the GOOGLE_API_KEY environment variable before running this notebook.")
google_api_key: SecretStr = SecretStr(os.environ["GOOGLE_API_KEY"])

## Step 1: Load the PDF Document

We'll use LangChain's PyPDFLoader to download and parse the "Alice in Wonderland" PDF.

In [17]:
pdf_url: str = "https://www.adobe.com/be_en/active-use/pdf/Alice_in_Wonderland.pdf"

# The constructor, not load(), downloads the PDF: it sends an HTTP GET with no
# timeout and saves the file in a temporary directory. That directory is only
# deleted when the loader object is garbage-collected.
loader: PyPDFLoader = PyPDFLoader(
    file_path=pdf_url,
    mode="page",
    extraction_mode="plain",
    extract_images=False,
)
pages: list[Document] = loader.load()

print(f"Loaded {len(pages)} pages from the PDF")

Loaded 105 pages from the PDF


## Step 2: Split into Chunks

Long documents need to be split into smaller chunks for effective retrieval. We use `RecursiveCharacterTextSplitter` which tries to split on natural boundaries (paragraphs, sentences) while respecting the chunk size.

Here we use chunk size of 1000 characters with a 200 characters overlap.

In [18]:
text_splitter: RecursiveCharacterTextSplitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    # Tried in order: paragraph break, line break, space, then any character.
    separators=["\n\n", "\n", " ", ""],
    is_separator_regex=False,
    # True keeps each separator at the start of the chunk that follows it.
    keep_separator=True,
    strip_whitespace=True,
    add_start_index=False,
)

chunks: list[Document] = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Split into 245 chunks


> **Note:** A chunk size of 1,000 characters with 200-character overlap is a common starting point for text splitting. The overlap ensures that sentences spanning a chunk boundary aren't lost. Smaller chunks improve retrieval precision but may lack context; larger chunks retain more context but may dilute relevance signals. Tune these values based on your document structure and query patterns.

## Step 3: Create Embeddings and Vector Store

We embed all chunks using Google's Gemini embedding model and store them in LanceDB, a serverless vector database with persistent storage.

In [26]:
import time

# --- REPLACE YOUR INGESTION LOOP PARAMETERS WITH THIS ---
BATCH_SIZE = 10         # Lowered from 50 to 5 to avoid crashing the 30,000 token limit
SLEEP_DURATION = 12.0  # Increased to 12 seconds to ensure strict 30k TPM compliance
vectorstore = None

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    
    if i == 0:
        vectorstore = LanceDB.from_documents(
            documents=batch,
            embedding=embedding_model,
            uri="/tmp/lancedb",
            table_name="vectorstore",
            mode="overwrite",
            distance="l2",
            api_key=None,
        )
    else:
        vectorstore.add_documents(batch)
        
    print(f"Successfully processed chunks {i} to {min(i + BATCH_SIZE, len(chunks))}")
    
    if i + BATCH_SIZE < len(chunks):
        # Explicit block to let the Google API window fully clear its tracking token count
        time.sleep(SLEEP_DURATION)

print(f"Created vector store with {len(chunks)} documents")
# -----------------------------------------------------------------


Successfully processed chunks 0 to 10
Successfully processed chunks 10 to 20
Successfully processed chunks 20 to 30
Successfully processed chunks 30 to 40
Successfully processed chunks 40 to 50
Successfully processed chunks 50 to 60
Successfully processed chunks 60 to 70
Successfully processed chunks 70 to 80
Successfully processed chunks 80 to 90
Successfully processed chunks 90 to 100
Successfully processed chunks 100 to 110
Successfully processed chunks 110 to 120
Successfully processed chunks 120 to 130
Successfully processed chunks 130 to 140
Successfully processed chunks 140 to 150
Successfully processed chunks 150 to 160
Successfully processed chunks 160 to 170
Successfully processed chunks 170 to 180
Successfully processed chunks 180 to 190
Successfully processed chunks 190 to 200
Successfully processed chunks 200 to 210
Successfully processed chunks 210 to 220
Successfully processed chunks 220 to 230
Successfully processed chunks 230 to 240
Successfully processed chunks 240 to

## Step 4: RAG Query - Retrieval + Generation

Using LCEL (LangChain Expression Language), we create a chain that:
1. Takes a question
2. Retrieves relevant documents
3. Formats them as context
4. Passes to the LLM for answering

In [27]:
CHAT_MODEL: str = "gemini-3-flash-preview"

llm: ChatGoogleGenerativeAI = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    api_key=google_api_key,
    # Google recommends 1.0 for Gemini 3 models: lower values can cause
    # looping or degraded answers.
    temperature=1.0,
    thinking_level="low",
    # None means no timeout: a stalled request waits forever.
    timeout=None,
    max_retries=6,
)
retriever: VectorStoreRetriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)


def format_documents_as_context(documents: list[Document]) -> str:
    """Join the text of the retrieved chunks, separated by a blank line."""
    page_contents: list[str] = []
    for document in documents:
        page_contents.append(document.page_content)

    context: str = "\n\n".join(page_contents)
    return context


prompt_text: str = """Answer the question based only on the following context:

{context}

Question: {question}"""
prompt_template: ChatPromptTemplate = ChatPromptTemplate.from_template(
    template=prompt_text,
    template_format="f-string",
)

# Each LCEL step is built with its Runnable class. The `|` operator would build
# the same chain, but it silently wraps dicts in RunnableParallel and plain
# functions in RunnableLambda.
format_context_step: RunnableLambda = RunnableLambda(format_documents_as_context)
retrieve_context_step: RunnableSequence = RunnableSequence(
    retriever,
    format_context_step,
)
question_passthrough_step: RunnablePassthrough = RunnablePassthrough()
prompt_inputs_step: RunnableParallel = RunnableParallel(
    context=retrieve_context_step,
    question=question_passthrough_step,
)
output_parser: StrOutputParser = StrOutputParser()

rag_chain: RunnableSequence = RunnableSequence(
    prompt_inputs_step,
    prompt_template,
    llm,
    output_parser,
)

In [28]:
questions: list[str] = [
    "Who is the main character and what happens at the beginning of the story?",
    "What did the Caterpillar ask Alice?",
    "Describe the Mad Hatter's tea party.",
    "What happened at the trial at the end of the story?",
]
answer_separator_line: str = "-" * 60

for question in questions:
    answer: str = rag_chain.invoke(question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(answer_separator_line)

Q: Who is the main character and what happens at the beginning of the story?
A: Based on the context provided, it is not possible to identify the name of the main character or what happens at the beginning of the story.

The text provided is the conclusion of the book (noted by "THE END") and focuses on a character (referred to only as "she") picturing the future of her "little sister." The text mentions a "dream of Wonderland," but does not describe the start of the narrative.
------------------------------------------------------------
Q: What did the Caterpillar ask Alice?
A: Based on the context provided, there is no mention of the Caterpillar or any question asked by it. The text focuses on Alice's sister's reflections and the technical production details of the book.
------------------------------------------------------------
Q: Describe the Mad Hatter's tea party.
A: Based on the provided context, there is no description of the Mad Hatter’s tea party. The text mentions the snee